In [1]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
from dotenv import load_dotenv
load_dotenv()
API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
# from google.colab import userdata
# API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")


Client ready.


Section 1 — Talking to an LLM Programmatically

In [2]:
# Part 1.1 — Your first API call

# TODO: Write a helper function you will reuse for the WHOLE lab:

def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
             temperature=0.7, max_tokens=500):
     response = client.chat.completions.create(
         model=MODEL,
         messages=[
             {"role": "system", "content": system_prompt},
             {"role": "user",   "content": user_prompt},
         ],
         temperature=temperature,
         max_tokens=max_tokens,
     )
     return response.choices[0].message.content, response.usage


answer, usage = ask_llm("Who was the first man to be created?")
print(answer)
print(usage)



According to various religious and mythological traditions, the story of the first man created varies. Here are a few examples:

1. **Biblical account (Abrahamic religions)**: In the biblical account, the first man created is Adam, formed by God from the dust of the earth (Genesis 2:7). Adam is considered the first human being in the biblical account.
2. **Hindu mythology**: In Hindu mythology, the first man is said to be Manu, who was created by the god Brahma. Manu is considered the first human being and the father of humanity.
3. **Greek mythology**: In Greek mythology, the first humans were created by the gods, with the first man being Prometheus, who was created by the titan Epimetheus. However, the first human in the classical sense is often considered to be Deucalion, who survived a great flood and became the ancestor of the Greek people.
4. **Norse mythology**: In Norse mythology, the first humans were Ask and Embla, who were created by the gods from two pieces of driftwood.

I

Student Reasoning — Anatomy of a call 1. What is the difference between the system and user roles? Give an example of something that belongs in each. 2. What is a token, roughly? Why do API providers bill per token rather than per request?

In [3]:
# Part 1.2 — Temperature: the randomness dial

# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."

# TODO: Print all 10 answers, grouped by temperature.
question = "The names of the top games in 2026"

print("Temperature = 0.0")
for i in range(5):
    answer, usage = ask_llm(question, temperature=0.0)
    print(f"Answer {i+1}: {answer}\n")

print("Temperature = 1.2")
for i in range(5):
    answer, usage = ask_llm(question, temperature=1.2)
    print(f"Answer {i+1}: {answer}\n")

Temperature = 0.0
Answer 1: Since my knowledge cutoff is 2023, I don't have real-time information on the top games of 2026. However, I can give you an idea of the popular games that were trending in 2023, and you can use that as a starting point to explore the current gaming landscape.

Some of the popular games in 2023 included:

1. **Elden Ring** (Action RPG)
2. **Call of Duty: Modern Warfare II** (First-person shooter)
3. **God of War Ragnarök** (Action-adventure)
4. **The Last of Us Part I** (Action-adventure)
5. **Hogwarts Legacy** (Action RPG)
6. **Overwatch 2** (Team-based first-person shooter)
7. **Pokémon Scarlet and Violet** (Role-playing game)
8. **Warzone 2.0** (Battle royale)
9. **Destiny 2: The Witch Queen** (First-person shooter with MMO elements)
10. **Horizon Forbidden West** (Action RPG)

To find the top games of 2026, I recommend checking out online gaming communities, such as:

* Steam
* Xbox
* PlayStation
* Nintendo
* IGN
* GameSpot
* Polygon

These platforms often

Student Reasoning — Temperature What did you observe at each temperature? For the loan decision-support system you are about to build, which temperature regime is appropriate, and why?

Section 2 — The Dataset: Loan Application Letters

In [ ]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")



6 letters loaded.


Section 3 — Prompt Engineering for the Decision Support System

V1 on L002
Kwame Boateng, a commercial driver in Kumasi, is seeking a loan of GHS 25,000 to repair his vehicle's engine and pay off personal debts. He expects his business to improve after the festive season and is willing to repay the loan when he can, despite not having collateral at the moment.

V1 on L006
Kofi, a 22-year-old, is requesting a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no experience or collateral, but claims to be "business-minded" and promises to repay the loan within a year when his businesses are successful, relying on his trustworthiness.

V2 on L002
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle personal debts. Mr. Boateng mentions that business has been slow, but he expects it to improve after the festive season, and he is willing to repay the loan when he can. He does 

Student Reasoning — Summarization prompts 1. What concrete problems did V1's output have that V2 fixed? Quote examples. 2. Why is "no invented details" an essential instruction in this application? What is this failure mode called in the LLM literature?

Student Reasoning — Structured extraction 1. Why must the few-shot example NOT come from the six letters you are processing? 2. Why "use null, do not guess" — what did the model do without that instruction? 3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?